# KagMCP — servidor MCP para Kaggle

**KagMCP** es un servidor no oficial de Model Context Protocol para conectar clientes autorizados a Kaggle.

Incluye HTTPS con OAuth y PIN, herramientas MCP para archivos, proyectos, actividad, GPU e integraciones opcionales con GitHub y Hugging Face.

No requiere GPU ni Kaggle Secrets.

## 1. Arquitectura

Cliente MCP → HTTPS Gradio (respaldo Cloudflare) → Kaggle:127.0.0.1:8181/mcp

El binario Go proporciona herramientas MCP, un panel web seguro y métricas. El transporte HTTP local escucha solo en loopback y el estado privado se almacena en `/kaggle/working/.kagmcp`.

## 2. Clonar el repositorio remoto y compilar en Kaggle

No necesitas subir un ejecutable. **Ejecuta las siguientes tres celdas en orden** con Internet habilitado: clonar/actualizar `main` desde GitHub (sin sobrescribir cambios locales), comprobar/descargar Go y compilar el binario directamente en `/kaggle/working`. Si Go no está instalado o es anterior al mínimo de `go.mod`, se descarga de **go.dev** con comprobación del SHA-256 publicado por el equipo de Go. El binario se reconstruye cada vez que ejecutas la celda de compilación.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import tempfile
import urllib.request

REPO_URL = "https://github.com/ThowiLabs/kagssh-go.git"
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "kagssh-go"
BINARY = WORKING / "kagmcp-linux-amd64"

if platform.system() != "Linux" or platform.machine().lower() not in ("x86_64", "amd64"):
    raise RuntimeError("Este ejemplo requiere Kaggle Linux x86_64 (amd64).")
if not WORKING.is_dir():
    raise RuntimeError("Ejecuta este notebook en Kaggle (/kaggle/working).")
if not shutil.which("git"):
    raise RuntimeError("No se encontró git en el entorno de Kaggle.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True, timeout=180,
    )
else:
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} existe pero no es un clon Git. Muévelo antes de continuar.")
    current_origin = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip().rstrip("/")
    if current_origin.removesuffix(".git").lower() != REPO_URL.removesuffix(".git").lower():
        raise RuntimeError(f"El directorio ya existe y su remoto es distinto: {current_origin}")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("El clon tiene cambios sin guardar; no se actualizará ni se borrará nada.")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main"],
        check=True, timeout=180,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge", "--ff-only", "origin/main"],
        check=True, timeout=60,
    )

REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Repositorio listo:", REPO_DIR)
print("Revisión remota clonada:", REVISION)

In [ ]:
# Usa el Go del sistema si cumple el mínimo de go.mod; si falta o es antiguo,
# instala la versión necesaria desde la distribución oficial con SHA-256 verificado.
mod_text = (REPO_DIR / "go.mod").read_text(encoding="utf-8")
match = re.search(r"(?m)^go\s+(\d+)\.(\d+)(?:\.(\d+))?\s*$", mod_text)
if not match:
    raise RuntimeError("No se encontró la versión mínima de Go en go.mod.")
MIN_GO = tuple(int(v or 0) for v in match.groups())
required = ".".join(str(v) for v in MIN_GO)
go_executable = shutil.which("go")

def installed_version(executable):
    if not executable:
        return None
    result = subprocess.run([executable, "version"], capture_output=True, text=True)
    found = re.search(r"go(\d+)\.(\d+)\.(\d+)", result.stdout)
    return tuple(map(int, found.groups())) if result.returncode == 0 and found else None

existing = installed_version(go_executable)
# Go 1.27.0 y 1.27.1 preceden a los parches del 8/oct/2026.
def secure_go(version):
    return (version is not None and version >= MIN_GO
            and (version[:2] != (1, 27) or version[2] >= 2))

if secure_go(existing):
    GO_BIN = Path(go_executable)
    print("Go instalado suficiente:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())
else:
    if not os.environ.get("KAGGLE_USER_SECRETS_TOKEN") and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        print("Nota: esta instalación necesita acceso a Internet desde Kaggle.")
    release_version = "go" + required
    catalog_url = "https://go.dev/dl/?mode=json&include=all"
    with urllib.request.urlopen(catalog_url, timeout=40) as response:
        releases = json.load(response)
    release = next((item for item in releases if item.get("version") == release_version), None)
    if release is None:
        raise RuntimeError(f"La versión {release_version} no figura en las descargas oficiales.")
    archive_meta = next(
        (f for f in release["files"] if f.get("os") == "linux"
         and f.get("arch") == "amd64" and f.get("kind") == "archive"), None
    )
    if archive_meta is None or not re.fullmatch(r"[0-9a-f]{64}", archive_meta.get("sha256", "")):
        raise RuntimeError("No se pudo obtener el SHA-256 oficial de Go linux-amd64.")
    tempdir = Path(tempfile.mkdtemp(prefix="kagssh-toolchain-", dir="/tmp"))
    archive = tempdir / archive_meta["filename"]
    address = "https://go.dev/dl/" + archive_meta["filename"]
    print("Descargando Go verificado:", archive_meta["filename"])
    digest = hashlib.sha256()
    with urllib.request.urlopen(address, timeout=120) as source, archive.open("wb") as target:
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            digest.update(chunk)
    if digest.hexdigest() != archive_meta["sha256"]:
        archive.unlink(missing_ok=True)
        raise RuntimeError("SHA-256 inválido: se rechaza la instalación.")
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(tempdir)], check=True, timeout=120)
    archive.unlink(missing_ok=True)
    GO_BIN = tempdir / "go" / "bin" / "go"
    if not GO_BIN.is_file() or installed_version(str(GO_BIN)) != MIN_GO:
        raise RuntimeError("No se pudo validar la versión instalada de Go.")

print("Go seleccionado:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())
print("Versión mínima del repositorio:", required)

In [ ]:
# Compila el código clonado para el runtime Linux de Kaggle (sin CGO).
build_env = os.environ.copy()
build_env.update({"CGO_ENABLED": "0", "GOOS": "linux", "GOARCH": "amd64", "GOTOOLCHAIN": "local"})
subprocess.run(
    [str(GO_BIN), "build", "-buildvcs=false", "-trimpath", "-ldflags=-s -w",
     "-o", str(BINARY), "./cmd/kagssh"],
    cwd=str(REPO_DIR), env=build_env, check=True, timeout=600,
)
version = subprocess.run([str(BINARY), "-version"], capture_output=True, text=True, check=True)
print("Compilación terminada:", BINARY)
print("Versión binaria:", version.stdout.strip())
print("Go usado:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())

## 3. Configurar KagMCP en Python

Edita el túnel HTTPS, el puerto, el PIN y las integraciones opcionales. Si dejas `pin=""`, Go genera un PIN temporal y lo muestra una sola vez al iniciar.

La configuración normal se entrega por variables de entorno. Los tokens de integraciones se solicitan mediante `getpass`.

In [ ]:
# KAGMCP: CONFIGURACIÓN DEL SERVIDOR MCP
import getpass
import os

MCP_ENABLED = True
MCP_TUNNEL = "auto"
GRADIO_RETRIES = 3
MCP_PUBLIC_URL = ""
MCP_LISTEN_PORT = 8181
USE_GITHUB = False
USE_HUGGINGFACE = False

# Copia privada del entorno de ejecución, sin credenciales internas del kernel.
RUNTIME_ENV = {
    k: v for k, v in os.environ.items()
    if not (k.startswith("MCP_") or k.startswith("GITHUB_") or k == "HF_TOKEN")
}
RUNTIME_ENV.pop("KAGGLE_USER_SECRETS_TOKEN", None)
RUNTIME_ENV.pop("KAGGLE_IAP_TOKEN", None)
RUNTIME_ENV.update({
    "MCP_ENABLED": str(MCP_ENABLED).lower(),
    "MCP_TUNNEL": MCP_TUNNEL,
    "MCP_GRADIO_RETRIES": str(GRADIO_RETRIES),
    "MCP_LISTEN_PORT": str(MCP_LISTEN_PORT),
    "MCP_PUBLIC_URL": MCP_PUBLIC_URL,
})

pin = ""
if pin and (not 6 <= len(pin) <= 128 or any(c in pin for c in "\r\n\x00")):
    raise ValueError("PIN inválido: usa de 6 a 128 caracteres.")
RUNTIME_ENV["MCP_ACCESS_PIN"] = pin
if not pin:
    print("Go generará un PIN temporal al iniciar.")
del pin

if USE_GITHUB:
    github_pat = getpass.getpass("GitHub PAT: ")
    if not github_pat.strip():
        raise ValueError("Falta token GitHub.")
    RUNTIME_ENV["GITHUB_TOKEN"] = github_pat
    del github_pat

if USE_HUGGINGFACE:
    hf_pat = getpass.getpass("Hugging Face token: ")
    if not hf_pat.strip():
        raise ValueError("Falta token Hugging Face.")
    RUNTIME_ENV["HF_TOKEN"] = hf_pat
    del hf_pat

print("KagMCP configurado:", MCP_TUNNEL, "| GitHub:", USE_GITHUB, "| Hugging Face:", USE_HUGGINGFACE)
print("Repositorio:", REPO_DIR, "| Runtime:", WORKING)


## 4. Credenciales y PIN

No se necesitan Kaggle Secrets para ejecutar KagMCP. `pin=""` permite que Go genere un PIN temporal; puedes indicar uno propio de 6 a 128 caracteres.

Los tokens de GitHub y Hugging Face son opcionales. Introduce los valores por `getpass` y no publiques logs que contengan secretos o el PIN.

## 5. Validar la configuración Python

Ejecuta primero **configurar** (`pin = ""` es válido y no solicita nada; Go generará el PIN durante `run`). La siguiente celda valida las opciones **sin volver a consultar Kaggle Secrets** y sin abrir túneles; recibe `RUNTIME_ENV` ya preparado.


In [ ]:
# Comprueba la configuración que Python preparó, sin abrir túneles ni leer Secrets.
if "RUNTIME_ENV" not in globals() or "MCP_ACCESS_PIN" not in RUNTIME_ENV:
    raise RuntimeError("Primero ejecuta la celda configurar.")
check = subprocess.run(
    [str(BINARY), "-check"],
    env=RUNTIME_ENV, cwd=str(WORKING),
    text=True,
    capture_output=True,
    timeout=180,
)
if check.returncode != 0:
    # Los errores de configuración no deben incluir contraseñas/PIN/PAT.
    raise RuntimeError(
        "Configuración inválida. Revisa las opciones de la celda configurar. "
        f"Detalle: {check.stderr.strip()[:600]}"
    )
print(check.stdout.strip())
print("Validado con el entorno Python; no se consultaron Kaggle Secrets adicionales.")


## 6. Iniciar KagMCP y mostrar logs en vivo

La siguiente celda utiliza **el mismo `RUNTIME_ENV`** que validaste. Go no volverá a leer Kaggle Secrets.

Cuando el túnel esté listo, verás la URL `https://...gradio.live/mcp` si funciona Gradio. Tras agotar los reintentos, el respaldo será `https://...trycloudflare.com/mcp`. Úsala en ChatGPT y autoriza con el PIN definido en `configurar` o el PIN temporal que Go muestra en los logs de `run`.

Para detenerlo, pulsa **Detener/Interruptar**: finaliza el proceso Go y sus túneles. Si modificas parámetros, vuelve a ejecutar **configurar → validación → run**.


In [ ]:
# Mantiene la celda de Kaggle en ejecución y muestra los logs de Go directamente.
# Usa Detener / Interruptar para cerrar KagMCP.
import subprocess

if "RUNTIME_ENV" not in globals() or "MCP_ACCESS_PIN" not in RUNTIME_ENV:
    raise RuntimeError("Primero ejecuta la celda configurar.")
print("Iniciando KagMCP. Pulsa Detener/Interruptar en Kaggle para desconectar.", flush=True)
with subprocess.Popen(
    [str(BINARY)],
    cwd=str(WORKING),
    env=RUNTIME_ENV,  # Entorno preparado en Python, sin token interno de Kaggle Secrets.
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    errors="replace",
    # Se reenvía cada línea a print() de Jupyter, sin hilos ni segundo plano.
) as kagssh:
    try:
        for line in kagssh.stdout:  # Bucle bloqueante: mantiene activa la celda.
            print(line, end="", flush=True)  # Logs visibles en la misma celda.
        status = kagssh.wait()
    except KeyboardInterrupt:
        print("\nInterrupción de Kaggle: cerrando KagMCP...", flush=True)
        if kagssh.poll() is None:
            kagssh.terminate()
        try:
            kagssh.wait(timeout=10)
        except subprocess.TimeoutExpired:
            kagssh.kill()
            kagssh.wait()
        print("KagMCP detenido; túnel cerrado.", flush=True)
    else:
        if status != 0:
            raise RuntimeError(f"KagMCP terminó con error (código {status}). Revisa los logs anteriores.")
        print("KagMCP terminó normalmente.", flush=True)


## 7. Conectar ChatGPT o un cliente MCP

También verás `panel_web=https://...gradio.live/` (o `https://...trycloudflare.com/` cuando se use el respaldo). Abre esta URL raíz y usa el PIN de MCP para gestionar proyectos, memoria, tareas, historial, GitHub, Hugging Face, métricas de GPU/almacenamiento y la Skill Ponytail v2 siempre activa. Para cada proyecto, verifica primero repo Git y notebook; después crea su UI Gradio con versiones y dependencias bloqueadas. Vigila el disco para evitar readonly.

Cuando veas en los logs:

```text
KagMCP listo para conectar cliente MCP url=https://...gradio.live/mcp
```

Usa la URL HTTPS **completa** y autoriza OAuth con el PIN definido en `configurar` o el temporal generado por Go en los logs de `run`. Ese PIN **no** se lee desde Kaggle Secrets.



## 8. Herramientas disponibles

- `environment_info`: descubre el runtime Kaggle.
- `list_dir`, `read_file`, `write_file`: inspecciona y edita archivos. `/kaggle/input` es solo lectura; `/kaggle/working` admite escritura.
- `exec`: ejecuta comandos con límites de tiempo y salida, desde `/kaggle/working`.
- `github_status`, `github_repositories`, `github_branches`, `github_file_read`, `github_file_write`: consultas y commits GitHub con PAT opcional.

El acceso MCP es directo al runtime: después de autorizar, los comandos pueden ejecutarse con permisos elevados del proceso. Autoriza solo clientes de confianza.


## Límites y seguridad

- KagMCP es un servidor MCP independiente y no oficial para Kaggle.
- Conexión mediante HTTPS, OAuth y PIN; túnel Gradio con respaldo Cloudflare.
- Las herramientas MCP, incluida `exec`, operan con permisos del proceso, con límites y autorización.
- `/kaggle/input` es de solo lectura y `/kaggle/working` contiene archivos modificables.
- El estado privado del servidor se guarda en `/kaggle/working/.kagmcp`.
- No publiques PIN ni tokens de integraciones.

Documentación: `README.md` y `USO_Y_CONFIGURACION.md`.